# Lecture 03: Machine Learning Basics, EDA and Feature Engineering

This notebook walks through the steps that come before you train a model. You explore the data, check correlations, engineer features, scale them and split the data. It uses the same `credit_risk_dataset.csv` your project works with.

Keep this notebook as a reference. You will do the same steps on your own data for the semester project. The notebook `02_Training_and_Metrics.ipynb` continues where this one ends.

In [ ]:
!uv pip install pandas matplotlib seaborn scikit-learn fg-data-profiling

### Load the data

In [ ]:
import numpy as np
import pandas as pd

BLUE, PINK, SLATE = "#77C5D8", "#E5007D", "#466873"

df = pd.read_csv("../../data/credit_risk_dataset.csv")
df.head()

## Exploratory Data Analysis (EDA)

### First look: shape, types, ranges

In [ ]:
df.info()

In [ ]:
df.describe()

### Missing values and duplicates

Always check both before you do anything else. This dataset has no missing values, but do not assume that. Check it explicitly, every time.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Remove duplicates if you find any
df = df.drop_duplicates()
print("Duplicate rows after removal:", df.duplicated().sum())

### Class balance

Check how often the target occurs. This benchmark is balanced on purpose, with about as many defaults as non-defaults. Real loan books have far fewer defaults. Notebook 02 shows why that matters for the choice of metric.

In [ ]:
df["dlq_2yrs"].value_counts(normalize=True).round(3)

### Distribution and outliers

`monthly_inc` is right-skewed. `rev_util` (revolving utilisation) is a ratio of balance to credit limit, so it should lie roughly between 0 and 1.

In [ ]:
import matplotlib.pyplot as plt

df["monthly_inc"].clip(upper=25_000).plot.hist(bins=40, color=BLUE, title="Monthly income (clipped at 25k)")
plt.show()

In [ ]:
df["rev_util"].plot.box(logy=True, title="Revolving utilisation (log scale)")
plt.show()
print(f"share of rows with rev_util above 1: {(df['rev_util'] > 1).mean():.2f}")

### Codes that look like counts

`late_30_59`, `late_60_89` and `late_90` count how often an applicant paid late, in three delay bands. Look at their largest values.

In [ ]:
late_cols = ["late_30_59", "late_60_89", "late_90"]
for c in late_cols:
    print(c, df[c].value_counts().sort_index().tail(4).to_dict())

In [ ]:
counts = df["late_30_59"].value_counts().sort_index()
colours = [PINK if v in (96, 98) else BLUE for v in counts.index]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar([str(int(v)) for v in counts.index], counts.values, color=colours)
ax.set_yscale("log")
ax.set_xlabel("late_30_59: times 30 to 59 days late")
ax.set_ylabel("applicants (log scale)")
plt.show()

Real counts run from 0 to 17. A few rows hold 96 or 98, in all three columns at once. Nobody is late 98 times in two years. These are very likely codes from the original data source, not counts. Treated as numbers, they distort every mean, every scaler and every correlation.

In [ ]:
coded = df[late_cols].isin([96, 98]).any(axis=1)
print(f"rows with a code: {int(coded.sum())} ({coded.mean():.2%} of all rows)")
print(f"codes in all three columns of those rows: {df.loc[coded, late_cols].isin([96, 98]).all(axis=1).all()}")
print(f"default rate in coded rows: {df.loc[coded, 'dlq_2yrs'].mean():.1%}, in all rows: {df['dlq_2yrs'].mean():.1%}")

The simplest fix drops these rows. We keep the original as `df_raw` for comparison and continue with the cleaned `df`.

Dropping is not free. The coded rows default far more often than the rest. Keeping them with a flag column, such as `late_code = coded.astype(int)`, is the alternative. Decide how your project treats them and record the decision in your decision log.

In [ ]:
df_raw = df.copy()
df = df[~coded].copy()
print(f"before: {len(df_raw)} rows, after: {len(df)} rows")
print("largest values now:", {c: int(df[c].max()) for c in late_cols})

### Automated EDA with fg-data-profiling

Checking columns one by one works. For a first pass on an unfamiliar dataset, a one-line profiling report is faster. `ydata-profiling` was recently renamed to `fg-data-profiling`. The tool is the same, only the package and import names changed.

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, title="Credit Risk Dataset: Profiling Report", minimal=True, progress_bar=False)
profile.to_file("credit_risk_profile.html")
print("Report written to credit_risk_profile.html. Open it in a browser.")

## Correlation


### The correlation matrix

For the full matrix we use Spearman's rho. Most features are skewed and some still hold extreme values after the cleaning step. Spearman works on ranks, so these values hardly move it.

In [ ]:
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

corr = df.corr(method="spearman")
cmap = LinearSegmentedColormap.from_list("course", [PINK, "#FFFFFF", BLUE])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap=cmap, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
ax.set_title("Spearman correlation between features")
plt.tight_layout()
plt.show()

### What rho misses

Spearman's rho only sees relationships that go in one direction. In the matrix, `open_credit` and the target show a rho close to 0. Plot the default rate per number of open credit lines and look at the shape. Applicants with very few lines default most, the risk is lowest around eight lines and rises again after that. A U-shape like this hides behind a rho near 0, so plot a feature before you call it unrelated.

In [ ]:
rate = df.groupby(df["open_credit"].clip(upper=25).astype(int))["dlq_2yrs"].mean()
print(f"Spearman rho of open_credit and the target: {df['open_credit'].corr(df['dlq_2yrs'], method='spearman'):.2f}")

ax = rate.plot.bar(color=BLUE, figsize=(9, 3.2))
ax.set_xlabel("open credit lines (25 means 25 or more)")
ax.set_ylabel("default rate")
plt.tight_layout()
plt.show()

## Feature Engineering

Our dataset already contains engineered features. `debt_ratio` divides debt by income, and `rev_util` divides the balance by the credit limit. Below are two more quick, domain-informed features. The first combines the three late-payment counts into one signal. The second turns income into a per-dependent figure. Without the cleaning step, the coded rows would get a sum of almost 300 late payments.

In [ ]:
df_fe = df.copy()
df_fe["total_late_payments"] = df_fe[late_cols].sum(axis=1)
df_fe["income_per_dependent"] = df_fe["monthly_inc"] / (df_fe["dependents"] + 1)

df_fe[["total_late_payments", "income_per_dependent"]].describe()

## Feature Scaling and Normalisation

Models that use distances or gradients are sensitive to the scale of each feature. Standardisation is also called z-score scaling.

| Method | Formula |
|---|---|
| Min-max scaling | $x' = \dfrac{x - \min(x)}{\max(x) - \min(x)}$ |
| Standardisation (z-score) | $x' = \dfrac{x - \mu}{\sigma}$ |
| Robust scaling | $x' = \dfrac{x - \operatorname{median}(x)}{\operatorname{IQR}(x)}$ |

One applicant earns 250,000 a month. Fit each scaler twice on `monthly_inc`: once on the raw values and once after clipping at the 99th percentile. The boxes show where the typical 99% of applicants land. Which scaler gives the same result with and without the extreme values?

Here we fit on all rows to compare the scalers. In your pipeline, learn the clip limit and the scaler on the training set only.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

income = df[["monthly_inc"]].to_numpy()
cap = np.percentile(income, 99)
clipped = np.clip(income, None, cap)
typical = income.ravel() <= cap   # the 99% of applicants that clipping does not change
print(f"max income: {income.max():,.0f}, 99th percentile: {cap:,.0f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (name, Scaler) in zip(axes, [("min-max", MinMaxScaler), ("z-score", StandardScaler), ("robust", RobustScaler)]):
    on_raw = Scaler().fit(income).transform(income).ravel()[typical]
    on_clipped = Scaler().fit(clipped).transform(clipped).ravel()[typical]
    ax.boxplot([on_raw, on_clipped], orientation="horizontal", whis=(5, 95), showfliers=False)
    ax.set_yticks([1, 2], ["fitted on raw values", "fitted after clipping"] if name == "min-max" else ["", ""])
    ax.set_title(name)
    print(f"{name}: typical applicants move by up to {np.abs(on_raw - on_clipped).max():.2f} when the extreme values are clipped")
plt.tight_layout()
plt.show()

## Splitting the Data

Split into training, validation and test sets before you fit anything. Stratify on the target, so that all three sets keep the same class balance. Fix `random_state`, so that the split is the same every time you run the notebook.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c != "dlq_2yrs"]
X, y = df[feature_cols], df["dlq_2yrs"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

for name, split_y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:>5}: {len(split_y):>5} rows, {split_y.mean():.1%} positive")

## Avoid leakage: fit on the training set only

Fit the scaler on `X_train` only. Then use that same fitted scaler to transform `X_val` and `X_test`. Never fit it again on validation or test data. That would leak information about them into your model.

In [ ]:
scaler = StandardScaler().fit(X_train)            # learns mean and std from training data only
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)            # transform only
X_test_scaled = scaler.transform(X_test)          # transform only

print("training means after scaling:  ", X_train_scaled.mean(axis=0).round(2)[:4])
print("validation means after scaling:", X_val_scaled.mean(axis=0).round(2)[:4])

The training means are exactly 0. The validation means are close to 0, but not exactly. That small difference is expected. It shows that the scaler has not seen the validation data.